In [1]:
!apt-get install -y openmpi-bin openmpi-common libopenmpi-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
libopenmpi-dev set to manually installed.
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
openmpi-common is already the newest version (4.1.2-2ubuntu1).
openmpi-common set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
%%writefile mpi_kmeans.cpp

#include <mpi.h>
#include <iostream>
#include <vector>
#include <cmath>
#include <cstdlib>
#include <ctime>

using namespace std;

#define POINTS 12
#define K 2
#define ITERATIONS 5

struct Point {
    float x, y;
};

// Distance Function
float distance(Point a, Point b) {
    return sqrt((a.x - b.x) * (a.x - b.x) +
                (a.y - b.y) * (a.y - b.y));
}

int main(int argc, char* argv[]) {

    MPI_Init(&argc, &argv);

    int rank, size;

    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    vector<Point> points;
    vector<Point> centroids(K);

    int chunk = POINTS / size;

    vector<Point> local_points(chunk);

    // Root Process Generates Dataset
    if (rank == 0) {

        srand(time(0));

        points.resize(POINTS);

        cout << "Dataset:\n";

        for (int i = 0; i < POINTS; i++) {
            points[i].x = rand() % 100;
            points[i].y = rand() % 100;

            cout << "(" << points[i].x
                 << ", " << points[i].y << ")\n";
        }

        // Initialize Centroids
        centroids[0] = points[0];
        centroids[1] = points[1];
    }

    // Broadcast Centroids
    MPI_Bcast(centroids.data(),
              K * sizeof(Point),
              MPI_BYTE,
              0,
              MPI_COMM_WORLD);

    // Scatter Dataset
    MPI_Scatter(rank == 0 ? points.data() : NULL,
                chunk * sizeof(Point),
                MPI_BYTE,
                local_points.data(),
                chunk * sizeof(Point),
                MPI_BYTE,
                0,
                MPI_COMM_WORLD);

    double start = MPI_Wtime();

    // K-Means Iterations
    for (int iter = 0; iter < ITERATIONS; iter++) {

        vector<float> sumX(K, 0);
        vector<float> sumY(K, 0);
        vector<int> count(K, 0);

        // Assign Points to Clusters
        for (int i = 0; i < chunk; i++) {

            float minDist = distance(local_points[i], centroids[0]);
            int cluster = 0;

            for (int j = 1; j < K; j++) {

                float dist = distance(local_points[i], centroids[j]);

                if (dist < minDist) {
                    minDist = dist;
                    cluster = j;
                }
            }

            sumX[cluster] += local_points[i].x;
            sumY[cluster] += local_points[i].y;
            count[cluster]++;
        }

        // Global Reduction
        vector<float> globalSumX(K);
        vector<float> globalSumY(K);
        vector<int> globalCount(K);

        MPI_Allreduce(sumX.data(), globalSumX.data(),
                      K, MPI_FLOAT, MPI_SUM,
                      MPI_COMM_WORLD);

        MPI_Allreduce(sumY.data(), globalSumY.data(),
                      K, MPI_FLOAT, MPI_SUM,
                      MPI_COMM_WORLD);

        MPI_Allreduce(count.data(), globalCount.data(),
                      K, MPI_INT, MPI_SUM,
                      MPI_COMM_WORLD);

        // Update Centroids
        for (int j = 0; j < K; j++) {

            if (globalCount[j] != 0) {
                centroids[j].x = globalSumX[j] / globalCount[j];
                centroids[j].y = globalSumY[j] / globalCount[j];
            }
        }
    }

    double end = MPI_Wtime();

    // Final Output
    if (rank == 0) {

        cout << "\nFinal Centroids:\n";

        for (int i = 0; i < K; i++) {
            cout << "Cluster " << i << ": ("
                 << centroids[i].x << ", "
                 << centroids[i].y << ")\n";
        }

        cout << "\nExecution Time: "
             << (end - start)
             << " sec\n";
    }

    MPI_Finalize();

    return 0;
}

Writing mpi_kmeans.cpp


In [3]:
!mpic++ mpi_kmeans.cpp -o mpi_kmeans

In [4]:
!mpirun --allow-run-as-root -np 1 ./mpi_kmeans

Dataset:
(3, 35)
(99, 7)
(20, 6)
(35, 47)
(81, 29)
(37, 42)
(29, 91)
(4, 57)
(8, 31)
(69, 25)
(11, 15)
(87, 71)

Final Centroids:
Cluster 0: (18.375, 40.5)
Cluster 1: (84, 33)

Execution Time: 2.2656e-05 sec
